In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import *

spark = (
    SparkSession.builder
    .appName("Joins")
    .master("local[*]")   
    .getOrCreate()
)

spark

In [2]:
emp = [
    (1, "Alice", 10),
    (2, "Bob", 20),
    (3, "Charlie", 30),
    (4, "David", None)
]

emp_df = spark.createDataFrame(emp, ["emp_id", "name", "dept_id"])

In [4]:
dept = [
    (10, "HR"),
    (20, "IT"),
    (40, "Finance")
]

dept_df = spark.createDataFrame(dept, ["dept_id", "dept_name"])

In [5]:
emp_df.show()
dept_df.show()

+------+-------+-------+
|emp_id|   name|dept_id|
+------+-------+-------+
|     1|  Alice|     10|
|     2|    Bob|     20|
|     3|Charlie|     30|
|     4|  David|   NULL|
+------+-------+-------+

+-------+---------+
|dept_id|dept_name|
+-------+---------+
|     10|       HR|
|     20|       IT|
|     40|  Finance|
+-------+---------+



In [6]:
#inner join joins both left and right table by using one matching column in both it gives data of matching column of both
emp_df.join(dept_df,"dept_id","inner").show()

+-------+------+-----+---------+
|dept_id|emp_id| name|dept_name|
+-------+------+-----+---------+
|     10|     1|Alice|       HR|
|     20|     2|  Bob|       IT|
+-------+------+-----+---------+



In [7]:
#LEFT JOIN 
#Take all records from LEFT table (emp)
#Match from right if available else null

emp_df.join(dept_df,"dept_id","left").show()



+-------+------+-------+---------+
|dept_id|emp_id|   name|dept_name|
+-------+------+-------+---------+
|     10|     1|  Alice|       HR|
|     20|     2|    Bob|       IT|
|     30|     3|Charlie|     NULL|
|   NULL|     4|  David|     NULL|
+-------+------+-------+---------+



In [8]:
emp_df.join(dept_df,"dept_id","right").show()

+-------+------+-----+---------+
|dept_id|emp_id| name|dept_name|
+-------+------+-----+---------+
|     10|     1|Alice|       HR|
|     20|     2|  Bob|       IT|
|     40|  NULL| NULL|  Finance|
+-------+------+-----+---------+



In [9]:
#FULL OUTER JOIN
#Take everything from both tables
emp_df.join(dept_df, "dept_id", "outer").show()

+-------+------+-------+---------+
|dept_id|emp_id|   name|dept_name|
+-------+------+-------+---------+
|   NULL|     4|  David|     NULL|
|     10|     1|  Alice|       HR|
|     20|     2|    Bob|       IT|
|     30|     3|Charlie|     NULL|
|     40|  NULL|   NULL|  Finance|
+-------+------+-------+---------+



In [10]:
#LEFT ANTI JOIN
#Return rows in LEFT that DO NOT MATCH
emp_df.join(dept_df, "dept_id", "left_anti").show()

+-------+------+-------+
|dept_id|emp_id|   name|
+-------+------+-------+
|     30|     3|Charlie|
|   NULL|     4|  David|
+-------+------+-------+



In [11]:
#LEFT SEMI JOIN
#Return rows in LEFT that MATCH (but no right columns)

emp_df.join(dept_df, "dept_id", "left_semi").show()

+-------+------+-----+
|dept_id|emp_id| name|
+-------+------+-----+
|     10|     1|Alice|
|     20|     2|  Bob|
+-------+------+-----+



In [12]:
# Employees
employees = [
    (1, "Alice", 10, None, 70000),
    (2, "Bob", 20, 1, 60000),
    (3, "Charlie", 30, 1, 90000),
    (4, "David", None, 2, 50000),
    (5, "Eva", 40, 2, 80000)
]

emp_df = spark.createDataFrame(employees,
    ["emp_id", "emp_name", "dept_id", "manager_id", "salary"]
)

# Departments
departments = [
    (10, "HR", "New York"),
    (20, "IT", "Chicago"),
    (30, "Finance", "Boston"),
    (50, "Admin", "Dallas")
]

dept_df = spark.createDataFrame(departments,
    ["dept_id", "dept_name", "location"]
)

In [13]:
customers = [
    (1, "Rahul"),
    (2, "Anita"),
    (3, "John"),
    (4, "Priya")
]

cust_df = spark.createDataFrame(customers,
    ["customer_id", "customer_name"]
)

orders = [
    (101, 1, 500),
    (102, 2, 300),
    (103, 5, 700),   # invalid customer
    (104, 1, 200)
]

orders_df = spark.createDataFrame(orders,
    ["order_id", "customer_id", "amount"]
)

In [14]:
products = [
    (1, "Laptop"),
    (2, "Mobile"),
    (3, "Tablet"),
    (4, "Monitor")
]

products_df = spark.createDataFrame(products,
    ["product_id", "product_name"]
)

sales = [
    (1, 2),
    (2, 5),
    (1, 3)
]

sales_df = spark.createDataFrame(sales,
    ["product_id", "qty"]
)

In [15]:
transactions = [
    (1, "2024-01-01", 500),
    (2, "2024-01-02", 300),
    (1, "2024-01-03", 200)
]

trans_df = spark.createDataFrame(transactions,
    ["customer_id", "order_date", "amount"]
)

returns = [
    (1, "2024-01-01", 100),
    (2, "2024-01-05", 50)
]

returns_df = spark.createDataFrame(returns,
    ["customer_id", "return_date", "refund_amt"]
)

In [16]:
sales_emp = [
    (1, 1000),
    (2, 1500),
    (3, 2000)
]

sales_df2 = spark.createDataFrame(sales_emp,
    ["emp_id", "sales_amt"]
)

staff = [
    (1, "Alice"),
    (2, "Bob"),
    (4, "David")
]

staff_df = spark.createDataFrame(staff,
    ["employee_id", "employee_name"]
)

In [17]:
source = [
    (1, "A"),
    (2, "B"),
    (3, "C")
]

target = [
    (2, "B"),
    (3, "C"),
    (4, "D")
]

source_df = spark.createDataFrame(source, ["id", "name"])
target_df = spark.createDataFrame(target, ["id", "name"])

In [21]:
emp_df.join(dept_df,"dept_id","inner").select("dept_name","emp_id").show()

+---------+------+
|dept_name|emp_id|
+---------+------+
|       HR|     1|
|       IT|     2|
|  Finance|     3|
+---------+------+



In [23]:
emp_df.join(dept_df,"dept_id","left_semi").show()

+-------+------+--------+----------+------+
|dept_id|emp_id|emp_name|manager_id|salary|
+-------+------+--------+----------+------+
|     10|     1|   Alice|      NULL| 70000|
|     20|     2|     Bob|         1| 60000|
|     30|     3| Charlie|         1| 90000|
+-------+------+--------+----------+------+



In [24]:
orders_df.join(cust_df,"customer_id","left_anti").show()

+-----------+--------+------+
|customer_id|order_id|amount|
+-----------+--------+------+
|          5|     103|   700|
+-----------+--------+------+



In [27]:
cust_df.join(orders_df,"customer_id","left_semi").show()

+-----------+-------------+
|customer_id|customer_name|
+-----------+-------------+
|          1|        Rahul|
|          2|        Anita|
+-----------+-------------+

